# 📊 YouTube Yorumları ile Türkçe Duygu Analizi (LSTM)
Bu çalışma, küçük bir Türkçe YouTube yorum veri seti üzerinde LSTM ile duygu analizi yapar.

In [1]:
!pip install -q tensorflow pandas scikit-learn

In [2]:
import pandas as pd

veri = {
    "yorum": [
        "Bu video harikaydı, çok beğendim!",
        "Berbat bir içerik olmuş, zaman kaybı.",
        "Gerçekten eğitici ve faydalıydı.",
        "Hiç beğenmedim, çok sıkıcı.",
        "Güzel anlatım, teşekkür ederim!",
        "Nefret ettim bu videodan.",
        "Mükemmel bir anlatım olmuş.",
        "Çok kötüydü, bir daha izlemem.",
        "Emeğinize sağlık, çok güzeldi.",
        "Bu ne ya, hiç beğenmedim."
    ],
    "etiket": [1, 0, 1, 0, 1, 0, 1, 0, 1, 0]
}

df = pd.DataFrame(veri)
df.head()

,yorum,etiket
0,"Bu video harikaydı, çok beğendim!",1
1,"Berbat bir içerik olmuş, zaman kaybı.",0
2,Gerçekten eğitici ve faydalıydı.,1
3,"Hiç beğenmedim, çok sıkıcı.",0
4,"Güzel anlatım, teşekkür ederim!",1


In [3]:
import re

def temizle(metin):
    metin = metin.lower()
    metin = re.sub(r"http\S+", "", metin)
    metin = re.sub(r"[^a-zçğıöşü\s]", "", metin)
    metin = re.sub(r"\s+", " ", metin).strip()
    return metin

df["yorum_temiz"] = df["yorum"].apply(temizle)
df[["yorum", "yorum_temiz", "etiket"]]

,yorum,yorum_temiz,etiket
0,"Bu video harikaydı, çok beğendim!",bu video harikaydı çok beğendim,1
1,"Berbat bir içerik olmuş, zaman kaybı.",berbat bir içerik olmuş zaman kaybı,0
2,Gerçekten eğitici ve faydalıydı.,gerçekten eğitici ve faydalıydı,1
3,"Hiç beğenmedim, çok sıkıcı.",hiç beğenmedim çok sıkıcı,0
4,"Güzel anlatım, teşekkür ederim!",güzel anlatım teşekkür ederim,1
5,Nefret ettim bu videodan.,nefret ettim bu videodan,0
6,Mükemmel bir anlatım olmuş.,mükemmel bir anlatım olmuş,1
7,"Çok kötüydü, bir daha izlemem.",çok kötüydü bir daha izlemem,0
8,"Emeğinize sağlık, çok güzeldi.",emeğinize sağlık çok güzeldi,1
9,"Bu ne ya, hiç beğenmedim.",bu ne ya hiç beğenmedim,0


In [4]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer(num_words=1000)
tokenizer.fit_on_texts(df["yorum_temiz"])

X = tokenizer.texts_to_sequences(df["yorum_temiz"])
X_pad = pad_sequences(X, maxlen=20)

y = df["etiket"].values

In [5]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model = Sequential()
model.add(Embedding(input_dim=1000, output_dim=64, input_length=20))
model.add(LSTM(64, dropout=0.2))
model.add(Dense(1, activation='sigmoid'))

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [6]:
model.fit(X_pad, y, epochs=10, batch_size=2, validation_split=0.2)

Epoch 1/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 4s 163ms/step - accuracy: 0.6000 - loss: 0.6947 - val_accuracy: 0.5000 - val_loss: 0.6911
Epoch 2/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.5000 - loss: 0.6883 - val_accuracy: 0.5000 - val_loss: 0.6901
Epoch 3/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.6833 - loss: 0.6863 - val_accuracy: 0.5000 - val_loss: 0.6888
Epoch 4/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.6333 - loss: 0.6836 - val_accuracy: 1.0000 - val_loss: 0.6873
Epoch 5/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 1.0000 - loss: 0.6753 - val_accuracy: 1.0000 - val_loss: 0.6853
Epoch 6/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 1.0000 - loss: 0.6674 - val_accuracy: 1.0000 - val_loss: 0.6824
Epoch 7/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 1.0000 - loss: 0.6591 - val_accuracy: 1.0000 - val_loss: 0.6782
Epoch 8/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 1.0000 - loss: 0.6509 - val_accuracy: 1.0000 - val_loss: 0.6729

In [7]:
def predict_yorum(yeni_yorum):
    temiz = temizle(yeni_yorum)
    seq = tokenizer.texts_to_sequences([temiz])
    pad = pad_sequences(seq, maxlen=20)
    tahmin = model.predict(pad)[0][0]
    return f"'{yeni_yorum}' → {'Pozitif' if tahmin > 0.5 else 'Negatif'} ({tahmin:.2f})"

print(predict_yorum("Bu video çok güzel olmuş"))
print(predict_yorum("Nefret ettim, kötüydü"))
print(predict_yorum("Harika anlatım hocam"))
print(predict_yorum("Zaman kaybıydı."))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 234ms/step
'Bu video çok güzel olmuş' → Pozitif (0.54)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
'Nefret ettim, kötüydü' → Negatif (0.48)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
'Harika anlatım hocam' → Pozitif (0.57)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
'Zaman kaybıydı.' → Pozitif (0.52)
